# Data preprocessing for field validation

In [1]:
import sys
import pandas as pd  
import matplotlib.pyplot as plt
sys.path.append('../../../')   # Add parent directory to Python path
from utils.visualization import *

# Video data labels for ground truth

In [24]:
# read the label CSV file
df_label = pd.read_csv('../../../data/field_validation/grund_truth_labels.csv')
df_label

,Page,Label,start_timestamp,end_timestamp
0,1,1,"00:04:26,241","00:04:27,508"
1,2,5,"00:04:31,509","00:04:32,508"
2,3,1,"00:04:39,509","00:04:40,262"
3,4,5,"00:04:41,260","00:04:42,760"
4,5,4,"00:04:44,511","00:04:48,761"
5,6,1,"00:04:51,765","00:04:52,766"
6,7,5,"00:04:53,513","00:04:54,759"
7,8,1,"00:05:03,515","00:05:05,013"
8,9,7,"00:10:15,794","00:10:17,794"
9,10,1,"00:12:29,294","00:12:31,291"


In [25]:
df_video = pd.read_csv('../../../data/field_validation/video.csv')
df_video.head()

,NTP,Timestamp,Video,X,Y,Z
0,"2025-10-21, 15:15:42.0410","2025-10-21, 15:15:42.0390",00:00:00.104,-0.161850,-0.621307,-0.760971
1,"2025-10-21, 15:15:42.0420","2025-10-21, 15:15:42.0410",00:00:00.105,-0.166687,-0.623611,-0.760941
2,"2025-10-21, 15:15:42.0490","2025-10-21, 15:15:42.0480",00:00:00.112,-0.161774,-0.620071,-0.765106
3,"2025-10-21, 15:15:42.0520","2025-10-21, 15:15:42.0500",00:00:00.114,-0.156738,-0.621979,-0.765808
4,"2025-10-21, 15:15:42.0530","2025-10-21, 15:15:42.0520",00:00:00.116,-0.152634,-0.622070,-0.766769


In [28]:
# df_label, for each page, from start_timestamp	to end_timestamp, if column Video in df_video, in the range from  start_timestamp	to end_timestamp, then give the extra label from df_label
def assign_labels_to_video(df_video, df_label):
    """
    Assign labels from df_label to df_video based on timestamp ranges.
    
    Parameters:
    -----------
    df_video : pd.DataFrame
        DataFrame with video data containing a 'Video' timestamp column
    df_label : pd.DataFrame
        DataFrame with labels containing 'start_timestamp', 'end_timestamp', 
        and label columns
    
    Returns:
    --------
    pd.DataFrame
        df_video with additional label columns from df_label
    """
    # Create a copy to avoid modifying original
    df_result = df_video.copy()
    count = 0
    countmask = 0
    
    # Ensure timestamps are datetime objects
    df_result['Video'] = pd.to_datetime(df_result['Video'])
    df_label['start_timestamp'] = pd.to_datetime(df_label['start_timestamp'])
    df_label['end_timestamp'] = pd.to_datetime(df_label['end_timestamp'])
    
    # Get label columns (exclude timestamp columns)
    label_columns = [col for col in df_label.columns 
                     if col not in ['start_timestamp', 'end_timestamp']]
    
    # Initialize label columns in result DataFrame
    for col in label_columns:
        df_result[col] = None
    
    # Iterate through each row in df_label
    for idx, label_row in df_label.iterrows():
        start = label_row['start_timestamp']
        end = label_row['end_timestamp']
        
        # Find matching rows in df_video where video timestamp is in range
        mask = (df_result['Video'] >= start) & (df_result['Video'] <= end)
        
        # Assign labels to matching rows
        for col in label_columns:
            df_result.loc[mask, col] = label_row[col]
            count += 1
            countmask += mask.sum()

    print(f"Total labels assigned: {count}")
    print(f"Total rows updated: {countmask}")
    return df_result



In [29]:
# Apply the method
df_video_labeled = assign_labels_to_video(df_video, df_label)

C:\Users\liuzi\AppData\Local\Temp\ipykernel_14600\2387040338.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_result['Video'] = pd.to_datetime(df_result['Video'])


Total labels assigned: 94
Total rows updated: 51674
